# GRCh38: standard Umap versus sex-aware Umap

A basic descriptive analysis comparing **single-read mappability** from Umap/UCSC (unmasked GRCh38 reference) with the tracks generated in this project for the **XX** reference (the entire chrY is masked) and the **XY** reference (the chrY PARs are masked). The metric is the fraction of bases covered by at least one unique k-mer and is therefore bounded between 0 and 1.

By default, the notebook analyses **k=100**, a commonly used read length and the lightest exact UCSC track to query remotely. Set `K_VALUES = [24, 36, 50, 100]` to analyse every k-mer size shared by both sources. UCSC does not provide k=150 in this track. Local copies can be supplied through `UCSC_FILES`.

> Methodological note: UCSC distributes the single-read signal as a bigBed of uniquely mappable regions. The local tracks represent unique k-mer start positions. To make them comparable, every local run is extended by `k-2` bases, matching the convention observed in the UCSC bigBed, and overlapping intervals are then merged. Uncovered bases count as 0.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import pyBigWig

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 10})

# Locate the repository root when running from either notebooks/ or the repository root.
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p.resolve() for p in candidates if (p / "map/GRCh38").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find map/GRCh38. Run this notebook inside the repository.")

K_VALUES = [100]  # use [24, 36, 50, 100] for the full, slower comparison
CHROMS = [str(i) for i in range(1, 23)] + ["X", "Y"]
N_BINS = 200              # profile resolution per chromosome
FORCE_REFRESH = False     # True forces the cached summary table to be rebuilt

UCSC_BASE = "https://hgdownload.soe.ucsc.edu/gbdb/hg38/hoffmanMappability"
UCSC_FILES = {k: f"{UCSC_BASE}/k{k}.Unique.Mappability.bb" for k in K_VALUES}
# For offline use, replace URLs with local paths, for example:
# UCSC_FILES[100] = ROOT / "data/GRCh38/k100.Unique.Mappability.bb"

CORRECTED_FILES = {
    (sex, k): ROOT / f"map/GRCh38/GRCh38.{sex}/k{k}/GRCh38.{sex}.k{k}.single_read.bw"
    for sex in ("XX", "XY") for k in K_VALUES
}
CHROM_SIZES_FILE = ROOT / "refs/GRCh38/GRCh38.XY.chrom.sizes"
PAR_FILE = ROOT / "config/par/GRCh38.PAR.bed"
XY_REGIONS_FILE = ROOT / "config/xy_regions.txt"
OUTDIR = ROOT / "results/GRCh38_umap_comparison"
OUTDIR.mkdir(parents=True, exist_ok=True)
CACHE = OUTDIR / f"binned_single_read_{N_BINS}bins.csv.gz"

missing = [str(p) for p in [*CORRECTED_FILES.values(), CHROM_SIZES_FILE, PAR_FILE, XY_REGIONS_FILE] if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing local inputs:\n" + "\n".join(missing))
print(f"Repository root: {ROOT}")
print(f"Output directory: {OUTDIR}")
print(f"pyBigWig remote support: {bool(pyBigWig.remote)}")

## Reading and summarising the tracks

Each chromosome is divided into equal-width bins. Overlapping intervals are merged and the covered fraction is calculated within each bin; positions absent from the file count as 0. UCSC `chr1` and local `1` chromosome naming are harmonised automatically.

In [ ]:
def read_primary_sizes(path):
    sizes = {}
    with open(path) as handle:
        for line in handle:
            chrom, size = line.split()[:2]
            if chrom.removeprefix("chr") in CHROMS:
                sizes[chrom.removeprefix("chr")] = int(size)
    return sizes


def bw_chrom_name(bw, canonical):
    available = bw.chroms()
    for candidate in (canonical, f"chr{canonical}"):
        if candidate in available:
            return candidate
    return None


def interval_coverage_by_bin(bw, canonical, length, k, dilate_local=False, n_bins=N_BINS):
    edges = np.linspace(0, length, n_bins + 1, dtype=np.int64)
    widths = np.diff(edges).astype(float)
    bw_chrom = bw_chrom_name(bw, canonical)
    covered = np.zeros(n_bins, dtype=float)
    if bw_chrom is None:
        pass
    else:
        usable = min(length, bw.chroms()[bw_chrom])
        if usable != length:
            warnings.warn(f"{canonical}: track length={usable:,}, expected={length:,}")
        if bw.isBigBed():
            raw = bw.entries(bw_chrom, 0, usable) or []
            intervals = [(int(s), int(e)) for s, e, _ in raw]
        else:
            raw = bw.intervals(bw_chrom, 0, usable) or []
            extension = k - 2 if dilate_local else 0
            intervals = [(int(s), min(int(e) + extension, usable)) for s, e, value in raw if value > 0]
        # Merge overlaps; the UCSC bigBed contains overlapping regions.
        merged = []
        for start, end in sorted(intervals):
            if merged and start <= merged[-1][1]:
                merged[-1] = (merged[-1][0], max(merged[-1][1], end))
            else:
                merged.append((start, end))
        j = 0
        for i, (bin_start, bin_end) in enumerate(zip(edges[:-1], edges[1:])):
            while j < len(merged) and merged[j][1] <= bin_start:
                j += 1
            q = j
            while q < len(merged) and merged[q][0] < bin_end:
                covered[i] += max(0, min(bin_end, merged[q][1]) - max(bin_start, merged[q][0]))
                q += 1
    fractions = np.divide(covered, widths, out=np.zeros_like(covered), where=widths > 0)
    return edges[:-1], edges[1:], fractions


def summarize_track(source, label, k, chrom_sizes, dilate_local=False):
    try:
        bw = pyBigWig.open(str(source))
    except RuntimeError as exc:
        raise RuntimeError(
            f"Could not open {source}. Check the network or download the file and update UCSC_FILES[{k}]."
        ) from exc
    rows = []
    try:
        for chrom in CHROMS:
            starts, ends, values = interval_coverage_by_bin(
                bw, chrom, chrom_sizes[chrom], k=k, dilate_local=dilate_local
            )
            rows.extend({
                "k": k, "track": label, "chromosome": chrom, "bin": i,
                "start": int(start), "end": int(end), "mappability": float(value),
            } for i, (start, end, value) in enumerate(zip(starts, ends, values)))
    finally:
        bw.close()
    return rows


chrom_sizes = read_primary_sizes(CHROM_SIZES_FILE)
assert set(CHROMS) == set(chrom_sizes), "The size table does not contain all 24 primary chromosomes."

In [ ]:
if CACHE.exists() and not FORCE_REFRESH:
    bins = pd.read_csv(CACHE)
    print(f"Loaded cache: {CACHE}")
else:
    all_rows = []
    for k in K_VALUES:
        print(f"k={k}: UCSC standard")
        all_rows.extend(summarize_track(UCSC_FILES[k], "UCSC standard", k, chrom_sizes))
        for sex in ("XX", "XY"):
            print(f"k={k}: sex-aware {sex}")
            all_rows.extend(summarize_track(
                CORRECTED_FILES[(sex, k)], f"Sex-aware {sex}", k, chrom_sizes, dilate_local=True
            ))
    bins = pd.DataFrame(all_rows)
    bins.to_csv(CACHE, index=False)
    print(f"Wrote cache: {CACHE}")

bins.head()

## Summaries and figures

The first figure provides a chromosome-wide overview. The second quantifies the change relative to standard Umap in the autosomes, X and Y. The third shows where differences occur along X and Y for k=100, with the PARs shaded.

In [ ]:
bins["width"] = bins["end"] - bins["start"]
chrom_summary = (
    bins.assign(weighted=lambda d: d.mappability * d.width)
    .groupby(["k", "track", "chromosome"], as_index=False)
    .agg(signal_sum=("weighted", "sum"), bp=("width", "sum"))
)
chrom_summary["mean_mappability"] = chrom_summary.signal_sum / chrom_summary.bp
chrom_summary["chrom_order"] = chrom_summary.chromosome.map({c: i for i, c in enumerate(CHROMS)})
chrom_summary = chrom_summary.sort_values(["k", "track", "chrom_order"])
chrom_summary.to_csv(OUTDIR / "mean_mappability_by_chromosome.csv", index=False)
chrom_summary.head()

In [ ]:
colors = {"UCSC standard": "#4c4c4c", "Sex-aware XX": "#cc4c78", "Sex-aware XY": "#2878b5"}
fig, axes = plt.subplots(1, len(K_VALUES), figsize=(max(7, 5 * len(K_VALUES)), 4.2), squeeze=False, sharey=True, constrained_layout=True)
for ax, k in zip(axes.flat, K_VALUES):
    part = chrom_summary[chrom_summary.k == k]
    for track in colors:
        d = part[part.track == track]
        ax.scatter(d.chrom_order, d.mean_mappability, marker="_", lw=1.4, label=track, color=colors[track])
    ax.axvline(21.5, color="#999999", ls=":", lw=1)
    ax.set_title(f"k={k}")
    ax.set_ylim(-0.03, 1.03)
    ax.set_xticks(range(len(CHROMS)), CHROMS, rotation=90)
    ax.set_ylabel("Mappable fraction (single-read)")
axes[0, 0].legend(frameon=False, ncol=3, loc="lower left")
fig.suptitle("GRCh38: single-read mappable fraction by chromosome", fontsize=14)
fig.savefig(OUTDIR / "01_mean_mappability_by_chromosome.png", bbox_inches="tight")
fig.savefig(OUTDIR / "01_mean_mappability_by_chromosome.svg", bbox_inches="tight")
plt.show()

In [ ]:
territory_bins = bins.copy()
territory_bins["territory"] = np.where(territory_bins.chromosome.eq("X"), "chrX",
                                      np.where(territory_bins.chromosome.eq("Y"), "chrY", "Autosomes"))
territory = (territory_bins.assign(weighted=lambda d: d.mappability * d.width)
             .groupby(["k", "track", "territory"], as_index=False)
             .agg(signal_sum=("weighted", "sum"), bp=("width", "sum")))
territory["mean_mappability"] = territory.signal_sum / territory.bp
standard = territory[territory.track == "UCSC standard"][["k", "territory", "mean_mappability"]].rename(columns={"mean_mappability": "standard"})
delta = territory[territory.track != "UCSC standard"].merge(standard, on=["k", "territory"])
delta["delta"] = delta.mean_mappability - delta.standard
delta.to_csv(OUTDIR / "mean_mappability_and_delta_by_territory.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=False, constrained_layout=True)
for ax, territory_name in zip(axes, ["Autosomes", "chrX", "chrY"]):
    d = delta[delta.territory == territory_name]
    x = np.arange(len(K_VALUES)); width = 0.35
    for offset, track in [(-width/2, "Sex-aware XX"), (width/2, "Sex-aware XY")]:
        y = d[d.track == track].set_index("k").loc[K_VALUES, "delta"]
        ax.bar(x + offset, y, width, label=track, color=colors[track])
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x, K_VALUES)
    ax.set_xlabel("k-mer length")
    ax.set_title(territory_name)
    ax.set_ylabel("Δ mappable fraction vs UCSC")
axes[0].legend(frameon=False)
fig.suptitle("Effect of XX/XY correction relative to standard Umap", fontsize=14)
fig.savefig(OUTDIR / "02_delta_by_territory.png", bbox_inches="tight")
fig.savefig(OUTDIR / "02_delta_by_territory.svg", bbox_inches="tight")
plt.show()
delta[["k", "track", "territory", "mean_mappability", "standard", "delta"]].round(5)

In [ ]:
par = pd.read_csv(PAR_FILE, sep="\t", header=None, names=["chromosome", "start", "end", "name"])
par["chromosome"] = par.chromosome.str.removeprefix("chr")
xy_regions = pd.read_csv(XY_REGIONS_FILE, sep="\t", comment="#")
xtr = xy_regions[xy_regions.region.eq("XTR")].copy()
profile = bins[(bins.k == 100) & bins.chromosome.isin(["X", "Y"])].copy()
profile["mid_mb"] = (profile.start + profile.end) / 2e6

fig, axes = plt.subplots(2, 1, figsize=(14, 6.5), constrained_layout=True)
for ax, chrom in zip(axes, ["X", "Y"]):
    d = profile[profile.chromosome == chrom]
    for track in colors:
        q = d[d.track == track]
        ax.plot(q.mid_mb, q.mappability, lw=1.2, color=colors[track], label=track)
    for _, region in par[par.chromosome == chrom].iterrows():
        ax.axvspan(region.start / 1e6, region.end / 1e6, color="#f2c14e", alpha=0.28)
    if chrom == "X":
        for _, region in xtr.iterrows():
            ax.axvspan(region.start / 1e6, region.end / 1e6, color="#8e6bbf", alpha=0.24)
    ax.set_xlim(0, chrom_sizes[chrom] / 1e6)
    ax.set_ylim(-0.03, 1.03)
    ax.set_ylabel("Mappable fraction")
    shaded_regions = "shaded PARs + XTR" if chrom == "X" else "shaded PARs"
    ax.set_title(f"chr{chrom} (k=100; {shaded_regions})")
axes[-1].set_xlabel("Position (Mb)")
line_handles, line_labels = axes[0].get_legend_handles_labels()
region_handles = [
    Patch(facecolor="#f2c14e", alpha=0.28, label="PAR"),
    Patch(facecolor="#8e6bbf", alpha=0.24, label="XTR"),
]
axes[0].legend(line_handles + region_handles, line_labels + ["PAR", "XTR"], frameon=False, ncol=5)
fig.savefig(OUTDIR / "03_chrX_chrY_profiles_k100.png", bbox_inches="tight")
fig.savefig(OUTDIR / "03_chrX_chrY_profiles_k100.svg", bbox_inches="tight")
plt.show()

## Expected interpretation

- Autosomes should change very little because masking is restricted to chrY.
- In XX, chrY mappability should be 0 because the chromosome is completely masked.
- In both XX and XY, removing the competing Y PAR copies is expected to increase mappability in the X PARs.
- In XY, the Y PARs are 0 by construction while the remainder of Y remains available.

These are descriptive comparisons of regions covered by at least one unique k-mer. Differences between Umap/UCSC and the sex-aware tracks may also reflect the mappability engine and input FASTA, so the autosomal control helps assess the magnitude of that technical background.